In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np



## Align Neural, Behavioral, SLEAP Frames

In [3]:
def resolve_path(p) -> Path:
    clean = str(p).strip().strip('"').replace("\\", "/").lstrip("/")
    return lab_drive / clean

def load_paths(sheet_path: str | Path):

    sheet_df = pd.read_csv(sheet_path)

    beh_files = [resolve_path(p) for p in sheet_df["beh_csv_path"].tolist()]
    neu_files = [resolve_path(p) for p in sheet_df["neu_csv_path"].tolist()]
    sleap_files = [resolve_path(p) for p in sheet_df["sleap_csv_path"].tolist()]

    if not (len(beh_files) == len(neu_files) == len(sleap_files)):
        raise ValueError(f"File list lengths differ: beh={len(beh_files)} neu={len(neu_files)} sleap={len(sleap_files)}")

    missing = [p for p in beh_files + neu_files + sleap_files if not Path(p).exists()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} files not found (showing first 5): {missing[:5]}")

    return beh_files, neu_files, sleap_files


    return beh_paths_full, neu_paths_full, sleap_paths_full
    

In [4]:
def build_timeline(neu_df: pd.DataFrame,
    beh_df: pd.DataFrame, 
    fps: float, 
    ts_col: str = "Timestamp") -> pd.DataFrame:

    dt = pd.to_timedelta(1.0 / fps, unit="s")

    neu_ts = pd.to_datetime(neu_df[ts_col], utc=True, errors="coerce")
    beh_ts = pd.to_datetime(beh_df[ts_col], utc=True, errors="coerce")

    if neu_ts.isna().all() or beh_ts.isna().all():
        raise ValueError("All timestamps are NaT in neu or beh after parsing")

    t0 = min(neu_ts.min(), beh_ts.min())
    t1 = max(neu_ts.max(), beh_ts.max())

    global_ts = pd.date_range(start=t0, end=t1, freq=dt)

    return pd.DataFrame(
        {"global_idx": np.arange(len(global_ts), dtype=int), "global_ts": global_ts}
    )

def map_stream(stream_df: pd.DataFrame, 
    union_df: pd.DataFrame, 
    fps: float, 
    ts_col: str, 
    prefix: str, 
    add_frame_idx: bool = False,) -> pd.DataFrame:
    
    tol = pd.Timedelta(seconds=0.5 / fps)

    s = stream_df.copy()
    s[ts_col] = pd.to_datetime(s[ts_col], utc=True, errors="coerce")
    s = s.dropna(subset=[ts_col]).sort_values(ts_col).reset_index(drop=True)

    s["recorded_idx"] = np.arange(len(s), dtype=int)
    if add_frame_idx:
        s["beh_frame_idx"] = s["recorded_idx"]

    mapped = pd.merge_asof(
        union_df.sort_values("global_ts"),
        s.sort_values(ts_col),
        left_on="global_ts",
        right_on=ts_col,
        direction="nearest",
        tolerance=tol,
    )

    mapped[f"{prefix}_dropped"] = mapped["recorded_idx"].isna()
    mapped = mapped.rename(columns={ts_col: f"{prefix}_ts"})

    return mapped

def align_session(
    beh_path: Path,
    neu_path: Path,
    sleap_path: Path,
    fps: float,
    ts_col: str = "Timestamp",
    sleap_cols: list[str] | None = None,
) -> pd.DataFrame:

    beh_df = pd.read_csv(beh_path)
    neu_df = pd.read_csv(neu_path)
    sleap_df = pd.read_csv(sleap_path)

    timeline = build_timeline(neu_df, beh_df, fps=fps, ts_col=ts_col)

    beh_stream = map_stream(beh_df, timeline, fps=fps, ts_col=ts_col, prefix="beh", add_frame_idx=True)
    neu_stream = map_stream(neu_df, timeline, fps=fps, ts_col=ts_col, prefix="neu", add_frame_idx=False)

    aligned = beh_stream.merge(
        neu_stream.drop(columns=["global_ts"]),  
        on="global_idx",
        how="left",
        suffixes=("", "_neu"),
    )

    sleap = sleap_df.copy().rename(columns={"frame_idx": "beh_frame_idx"})
    sleap["beh_frame_idx"] = pd.to_numeric(sleap["beh_frame_idx"], errors="coerce")
    aligned["beh_frame_idx"] = pd.to_numeric(aligned["beh_frame_idx"], errors="coerce")

    if sleap_cols is None:
        sleap_cols = ["nose.x", "nose.y", "ear_L.x", "ear_L.y", "ear_R.x", "ear_R.y"]

    keep = ["beh_frame_idx"] + [c for c in sleap_cols if c in sleap.columns]
    aligned = aligned.merge(sleap[keep], on="beh_frame_idx", how="left")

    out_cols = ["global_idx", "global_ts", "beh_ts", "neu_ts", "beh_dropped", "neu_dropped", "beh_frame_idx"] + \
               [c for c in sleap_cols if c in aligned.columns]
    return aligned[out_cols]


def align_all(
    sheet_path: str | Path,
    fps: float,
    ts_col: str = "Timestamp",
) -> list[pd.DataFrame]:
    beh_files, neu_files, sleap_files = load_paths(sheet_path)

    aligned_sessions: list[pd.DataFrame] = []
    for session_id, (beh_p, neu_p, sleap_p) in enumerate(zip(beh_files, neu_files, sleap_files)):
        print(f"\n=== Session {session_id} ===")
        print("beh:", beh_p)
        print("neu:", neu_p)
        print("sleap:", sleap_p)

        aligned = align_session(
            beh_path=beh_p,
            neu_path=neu_p,
            sleap_path=sleap_p,
            fps=fps,
            ts_col=ts_col,
        )
        aligned["session_id"] = session_id
        aligned_sessions.append(aligned)

        bad = ((aligned["beh_dropped"] == True) & aligned.filter(like=".x").notna().any(axis=1)).sum()
        if bad:
            print(f"WARNING: {bad} rows have SLEAP coords on beh_dropped rows (check frame_idx meaning).")

    return aligned_sessions


In [5]:
sheet_path = "/Users/may/Projects/1p_pipeline/paths - Sheet1 (1).csv"
lab_drive = Path(os.environ.get("LAB_DRIVE_PATH", "/Volumes/ASA_Lab")) # replace with your actual lab drive path

fps = 30
aligned_sessions = align_all(sheet_path=sheet_path, fps=fps, ts_col="Timestamp")



=== Session 0 ===
beh: /Volumes/ASA_Lab/Data/May/Cross_Maze/CCTT6-1-12-A/20251028/beh-cam_frame-id_0.csv
neu: /Volumes/ASA_Lab/Data/May/Cross_Maze/CCTT6-1-12-A/20251028/miniscope_frame-id_0.csv
sleap: /Volumes/ASA_Lab/Users/May/SLEAP/Miniscope_Model/20260210_1-12-A_20251028.000_beh-cam_0.analysis.csv


In [6]:
aligned_sessions[0]

,global_idx,global_ts,beh_ts,neu_ts,beh_dropped,neu_dropped,beh_frame_idx,nose.x,nose.y,ear_L.x,ear_L.y,ear_R.x,ear_R.y,session_id
0,0,2025-10-28 21:03:43.801088+00:00,NaT,2025-10-28 21:03:43.801088+00:00,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,2025-10-28 21:03:43.834421333+00:00,NaT,2025-10-28 21:03:43.828211200+00:00,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2,2025-10-28 21:03:43.867754666+00:00,NaT,2025-10-28 21:03:43.860979200+00:00,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,3,2025-10-28 21:03:43.901087999+00:00,NaT,2025-10-28 21:03:43.894400+00:00,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,4,2025-10-28 21:03:43.934421332+00:00,NaT,2025-10-28 21:03:43.927180800+00:00,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53843,53843,2025-10-28 21:33:38.567736719+00:00,2025-10-28 21:33:38.578931200+00:00,2025-10-28 21:33:38.553011200+00:00,False,False,53828.0,1391.998291,1700.873901,1189.303345,1727.702759,1186.423950,1695.972168,0
53844,53844,2025-10-28 21:33:38.601070052+00:00,2025-10-28 21:33:38.612505600+00:00,2025-10-28 21:33:38.586790400+00:00,False,False,53829.0,1391.935059,1700.778687,1189.250854,1727.728027,1186.368286,1695.989990,0
53845,53845,2025-10-28 21:33:38.634403385+00:00,2025-10-28 21:33:38.645875200+00:00,2025-10-28 21:33:38.619571200+00:00,False,False,53830.0,1391.925049,1700.708008,1189.222412,1727.746094,1186.338623,1695.998901,0
53846,53846,2025-10-28 21:33:38.667736718+00:00,2025-10-28 21:33:38.679040+00:00,2025-10-28 21:33:38.652518400+00:00,False,False,53831.0,1391.927612,1700.758911,1189.250610,1727.759277,1186.359253,1696.002441,0


In [7]:
output = aligned_sessions[0].to_csv("/Users/may/Projects/1p_pipeline/aligned_session_0.csv", index=False)

## Select Arena ROIs

In [ ]:
import os, json
import numpy as np
import pandas as pd
import cv2


def collect_arena_roi_opencv(video_path, roi_json_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read first frame from: {video_path}")

    points = []
    window = "Arena ROI (L-click add, R-click undo, 's' save, 'q' quit)"

    def redraw(img, pts):
        out = img.copy()
        for (x, y) in pts:
            cv2.circle(out, (x, y), 4, (255, 0, 0), -1)
        if len(pts) >= 2:
            p = np.array(pts, np.int32).reshape((-1, 1, 2))
            cv2.polylines(out, [p], False, (0, 255, 0), 2)
        cv2.putText(out, f"Points: {len(pts)}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        return out

    display = redraw(frame, points)

    def on_mouse(event, x, y, flags, param):
        nonlocal points, display
        if event == cv2.EVENT_LBUTTONDOWN:
            points.append((int(x), int(y)))
            display = redraw(frame, points)
        elif event == cv2.EVENT_RBUTTONDOWN:
            if points:
                points.pop()
                display = redraw(frame, points)

    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(window, on_mouse)

    while True:
        cv2.imshow(window, display)
        key = cv2.waitKey(20) & 0xFF

        if key == ord('s'):
            if len(points) < 3:
                print("Need at least 3 points to save a polygon.")
                continue
            with open(roi_json_path, "w") as f:
                json.dump(points, f)
            print("Saved ROI:", roi_json_path)
            break

        if key == ord('q') or key == 27:
            cv2.destroyAllWindows()
            raise RuntimeError("Quit without saving ROI.")

    cv2.destroyAllWindows()
    return frame, points


def load_or_create_arena_mask(video_path, roi_json_path=None):
    if roi_json_path is None:
        roi_json_path = os.path.splitext(video_path)[0] + "_arena_roi.json"

    if os.path.exists(roi_json_path):
        with open(roi_json_path, "r") as f:
            arena_points = json.load(f)

        cap = cv2.VideoCapture(video_path)
        ret, frame0 = cap.read()
        cap.release()
        if not ret:
            raise RuntimeError(f"Could not read first frame from: {video_path}")

        print("Loaded ROI:", roi_json_path)
    else:
        frame0, arena_points = collect_arena_roi_opencv(video_path, roi_json_path)

    H, W = frame0.shape[:2]
    pts = np.array(arena_points, dtype=np.int32).reshape((-1, 1, 2))

    arena_mask = np.zeros((H, W), dtype=np.uint8)
    cv2.fillPoly(arena_mask, [pts], 255)

    return arena_mask, H, W, roi_json_path


def arena_mask_and_filter_positions(
    pose_csv_or_df,
    video_path,
    roi_json_path=None,
    ref_x="nose.x",
    ref_y="nose.y",
    pose_cols=None,
    add_in_arena_col=True,
    out_csv_path=None,
):
    """
    Creates/loads an arena mask from video ROI, then NaN-outs pose columns
    for rows whose (ref_x, ref_y) are outside the mask.

    pose_csv_or_df:
      - str path to CSV, OR a pandas DataFrame

    pose_cols:
      - list of columns to NaN-out when outside
      - if None, defaults to your usual pose set
    """

    # --- load df ---
    if isinstance(pose_csv_or_df, str):
        df = pd.read_csv(pose_csv_or_df)
    else:
        df = pose_csv_or_df.copy()

    if pose_cols is None:
        pose_cols = [
            "nose.x","nose.y",
            "ear_L.x","ear_L.y",
            "ear_R.x","ear_R.y",
            "ear_mid_x","ear_mid_y"
        ]

    # --- build/load arena mask ---
    arena_mask, H, W, roi_path_used = load_or_create_arena_mask(video_path, roi_json_path=roi_json_path)

    # --- compute inside/outside using ref point ---
    xs = df[ref_x].to_numpy()
    ys = df[ref_y].to_numpy()

    inside = np.zeros(len(df), dtype=bool)

    valid = np.isfinite(xs) & np.isfinite(ys)
    xi = xs[valid].astype(int)
    yi = ys[valid].astype(int)

    in_bounds = (xi >= 0) & (xi < W) & (yi >= 0) & (yi < H)
    valid_idx = np.where(valid)[0][in_bounds]

    inside[valid_idx] = arena_mask[yi[in_bounds], xi[in_bounds]] > 0

    if add_in_arena_col:
        df["in_arena"] = inside

    # --- NaN out pose cols outside arena ---
    existing_pose_cols = [c for c in pose_cols if c in df.columns]
    df.loc[~inside, existing_pose_cols] = np.nan

    # --- optional save ---
    if out_csv_path is not None:
        df.to_csv(out_csv_path, index=False)

    return df, arena_mask, roi_path_used